In [1]:
import pandas as pd
import pymysql
import os
import time

RAW_FILE = './2019-Oct.csv'
CLEAN_TEMP_FILE = 'cleaned_data_temp.csv'
CHUNK_SIZE = 500000  
DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': '123456',
    'database': 'ecommerce',
    'local_infile': 1
}

def step1_clean_and_buffer():

    print(f"Cleaning {RAW_FILE}...")
    if os.path.exists(CLEAN_TEMP_FILE):
        os.remove(CLEAN_TEMP_FILE)
        
    chunk_iter = pd.read_csv(RAW_FILE, chunksize=CHUNK_SIZE)
    total_rows = 0
    start_time = time.time()
    
    for i, chunk in enumerate(chunk_iter):
        chunk['category_code'] = chunk['category_code'].fillna('unknown')
        chunk['brand'] = chunk['brand'].fillna('unknown')
        
        chunk['event_time'] = chunk['event_time'].str.replace(' UTC', '')
        
        # append to temp file
        chunk.to_csv(CLEAN_TEMP_FILE, mode='a', header=False, index=False)
        
        rows = len(chunk)
        total_rows += rows
        print(f"   -> batch {i+1}: {rows} rows")
        
    
    print(f"Cleaning done. Time: {time.time() - start_time:.2f}s")
    print(f"Rows ready to load: {total_rows}")

def step2_bulk_load_mysql():
    """Bulk load the cleaned CSV into MySQL using LOAD DATA LOCAL INFILE.
    Automatically enables local_infile on server and client side.
    """
    print("Loading data into MySQL...")

    import pymysql
    from pymysql.constants import CLIENT

    # Absolute path of the cleaned CSV
    abs_path = os.path.abspath(CLEAN_TEMP_FILE)

    try:
        # Connect with local_infile enabled on client
        conn = pymysql.connect(
            host=DB_CONFIG['host'],
            user=DB_CONFIG['user'],
            password=DB_CONFIG['password'],
            database=DB_CONFIG['database'],
            local_infile=1,
            client_flag=CLIENT.LOCAL_FILES
        )
        cursor = conn.cursor()

        # Temporarily enable local_infile on the server
        try:
            cursor.execute("SET GLOBAL local_infile = 1;")
        except Exception as e:
            print(f"Warning: Could not set server local_infile=1: {e}")

        # Ensure we are using the correct database
        cursor.execute(f"USE {DB_CONFIG['database']};")

        sql = f"""
        LOAD DATA LOCAL INFILE '{abs_path}'
        INTO TABLE fact_events
        FIELDS TERMINATED BY ',' 
        ENCLOSED BY '"'
        LINES TERMINATED BY '\\n'
        (event_time, event_type, product_id, category_id, category_code, brand, price, user_id, user_session);
        """

        start_time = time.time()
        cursor.execute(sql)
        conn.commit()
        print(f"Data load completed in {time.time() - start_time:.2f}s")

    except Exception as e:
        print(f"Load failed: {e}")
        print("Hint: Make sure MySQL server and client allow local_infile.")

    finally:
        try:
            conn.close()
        except:
            pass


def step3_create_indexes():
 
    print("Creating indexes...")
    conn = pymysql.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    try:
        cursor.execute("CREATE INDEX idx_user ON fact_events(user_id);")
        cursor.execute("CREATE INDEX idx_event_time ON fact_events(event_time);")
        cursor.execute("CREATE INDEX idx_event_type ON fact_events(event_type);")
        print("Indexes created.")
    except Exception as e:
        print(f"Index creation skipped or failed: {e}")
        
    conn.commit()
    conn.close()

if __name__ == "__main__":
    if not os.path.exists(RAW_FILE):
        print(f"Error: {RAW_FILE} not found in this directory.")
    else:
        step1_clean_and_buffer()
        step2_bulk_load_mysql()
        step3_create_indexes()
        
        if os.path.exists(CLEAN_TEMP_FILE):
            os.remove(CLEAN_TEMP_FILE)
            print("Temp file removed. ETL complete.")


Cleaning ./2019-Oct.csv...
   -> batch 1: 500000 rows
   -> batch 2: 500000 rows
   -> batch 3: 500000 rows
   -> batch 4: 500000 rows
   -> batch 5: 500000 rows
   -> batch 6: 500000 rows
   -> batch 7: 500000 rows
   -> batch 8: 500000 rows
   -> batch 9: 500000 rows
   -> batch 10: 500000 rows
   -> batch 11: 500000 rows
   -> batch 12: 500000 rows
   -> batch 13: 500000 rows
   -> batch 14: 500000 rows
   -> batch 15: 500000 rows
   -> batch 16: 500000 rows
   -> batch 17: 500000 rows
   -> batch 18: 500000 rows
   -> batch 19: 500000 rows
   -> batch 20: 500000 rows
   -> batch 21: 500000 rows
   -> batch 22: 500000 rows
   -> batch 23: 500000 rows
   -> batch 24: 500000 rows
   -> batch 25: 500000 rows
   -> batch 26: 500000 rows
   -> batch 27: 500000 rows
   -> batch 28: 500000 rows
   -> batch 29: 500000 rows
   -> batch 30: 500000 rows
   -> batch 31: 500000 rows
   -> batch 32: 500000 rows
   -> batch 33: 500000 rows
   -> batch 34: 500000 rows
   -> batch 35: 500000 rows
  